In [ ]:
from datasets import load_dataset
import numpy as np

ds = load_dataset("gft/ttm4hvac-source-all-train")

d:\Promptathon\EnergyOptimizationEngine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = ds["train"].to_pandas()

In [3]:
display(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
display(df.head())
print("\nDescription:")
display(df.describe())
print("\nInformation:\n")
display(df.info())

'Shape: (1051200, 11)'


First 5 rows:


,time,Outdoor Air Temperature (C),Heating Setpoint (C),Cooling Setpoint (C),Room Air Temperature (C),Outdoor Humidity (%),Wind Speed (m/s),Direct Solar Radiation (W/m^2),HVAC Power Consumption (W),series_id,is_default
0,2019-01-01 00:00:00,3.150,13.073122,24.270164,18.945633,93.00,0.650,0.0,96.85838,0,False
1,2019-01-01 00:15:00,4.075,12.000000,24.921984,18.855467,93.50,0.325,0.0,127.47386,0,False
2,2019-01-01 00:30:00,5.000,12.000000,24.870577,18.773378,94.00,0.000,0.0,156.13360,0,False
3,2019-01-01 00:45:00,5.125,13.246161,24.631392,18.696230,93.75,0.000,0.0,183.73726,0,False
4,2019-01-01 01:00:00,5.250,13.090104,25.185616,18.624104,93.50,0.000,0.0,209.07198,0,False



Description:


,Outdoor Air Temperature (C),Heating Setpoint (C),Cooling Setpoint (C),Room Air Temperature (C),Outdoor Humidity (%),Wind Speed (m/s),Direct Solar Radiation (W/m^2),HVAC Power Consumption (W),series_id
count,1.051200e+06,1.051200e+06,1.051200e+06,1.051200e+06,1.051200e+06,1.051200e+06,1.051200e+06,1.051200e+06,1.051200e+06
mean,1.450559e+01,1.773202e+01,2.788590e+01,2.256387e+01,6.150977e+01,3.486216e+00,1.870510e+02,2.482349e+04,1.450000e+01
std,9.756779e+00,3.463221e+00,4.352277e+00,3.237074e+00,2.790830e+01,2.255729e+00,2.902756e+02,7.319196e+04,8.655446e+00
min,-1.390000e+01,1.000000e+01,2.250000e+01,8.769796e+00,3.000000e+00,0.000000e+00,0.000000e+00,2.000000e+01,0.000000e+00
25%,7.000000e+00,1.531710e+01,2.400000e+01,2.086114e+01,3.900000e+01,2.100000e+00,0.000000e+00,2.200000e+02,7.000000e+00
50%,1.302500e+01,1.771366e+01,2.670000e+01,2.297049e+01,6.675000e+01,3.100000e+00,0.000000e+00,1.580527e+03,1.450000e+01
75%,2.170000e+01,2.100000e+01,3.000000e+01,2.448307e+01,8.700000e+01,4.600000e+00,3.440000e+02,9.632874e+03,2.200000e+01
max,4.200000e+01,2.325000e+01,4.000000e+01,3.499247e+01,1.000000e+02,1.750000e+01,1.033000e+03,8.712076e+05,2.900000e+01



Information:

<class 'pandas.DataFrame'>
RangeIndex: 1051200 entries, 0 to 1051199
Data columns (total 11 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   time                            1051200 non-null  str    
 1   Outdoor Air Temperature (C)     1051200 non-null  float64
 2   Heating Setpoint (C)            1051200 non-null  float64
 3   Cooling Setpoint (C)            1051200 non-null  float64
 4   Room Air Temperature (C)        1051200 non-null  float64
 5   Outdoor Humidity (%)            1051200 non-null  float64
 6   Wind Speed (m/s)                1051200 non-null  float64
 7   Direct Solar Radiation (W/m^2)  1051200 non-null  float64
 8   HVAC Power Consumption (W)      1051200 non-null  float64
 9   series_id                       1051200 non-null  int64  
 10  is_default                      1051200 non-null  bool   
dtypes: bool(1), float64(8), int64(1), str(1)
memory usage: 100.

None

In [4]:
df.columns

Index(['time', 'Outdoor Air Temperature (C)', 'Heating Setpoint (C)',
       'Cooling Setpoint (C)', 'Room Air Temperature (C)',
       'Outdoor Humidity (%)', 'Wind Speed (m/s)',
       'Direct Solar Radiation (W/m^2)', 'HVAC Power Consumption (W)',
       'series_id', 'is_default'],
      dtype='str')

In [5]:
input_features = [
    'Outdoor Air Temperature (C)',
    'Outdoor Humidity (%)',
    'Wind Speed (m/s)',
    'Direct Solar Radiation (W/m^2)',
    'Room Air Temperature (C)',
    'Cooling Setpoint (C)',
    'Heating Setpoint (C)'
]
target_features = [
    'Room Air Temperature (C)',
    'HVAC Power Consumption (W)'
]

In [6]:
from sklearn.preprocessing import MinMaxScaler

input_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_scaled = input_scaler.fit_transform(
    df[input_features]
)

y_scaled = target_scaler.fit_transform(
    df[target_features]
)

In [7]:
grouped = df.groupby('series_id')

In [8]:
X_sequences = []
y_sequences = []

sequence_length = 24

for series_id, group in grouped:

    group = group.sort_values('time')

    X_group = input_scaler.transform(
        group[input_features]
    )

    y_group = target_scaler.transform(
        group[target_features]
    )

    for i in range(sequence_length, len(group)):

        X_sequences.append(
            X_group[i-sequence_length:i]
        )

        y_sequences.append(
            y_group[i]
        )

In [9]:
split = int(0.8 * len(X_sequences))

X_train = X_sequences[:split]
X_test = X_sequences[split:]

y_train = y_sequences[:split]
y_test = y_sequences[split:]

In [10]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

C:\Users\abhij\AppData\Local\Temp\ipykernel_8792\3247396029.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  X_train = torch.tensor(X_train, dtype=torch.float32)


In [12]:
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

X_train: torch.Size([840384, 24, 7])
y_train: torch.Size([840384, 2])
X_test: torch.Size([210096, 24, 7])
y_test: torch.Size([210096, 2])


In [16]:
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

In [17]:
from torch.utils.data import DataLoader

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [18]:
import torch.nn as nn

class HVACLSTM(nn.Module):

    def __init__(self):

        super(HVACLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=7,
            hidden_size=64,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(
            64,
            2
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out

In [19]:
model = HVACLSTM()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
criterion = nn.MSELoss()

In [ ]:
epochs = 50

train_losses = []

for epoch in range(epochs):

    model.train()

    epoch_loss = 0

    for batch_X, batch_y in train_loader:

        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)

    train_losses.append(avg_loss)

    print(
        f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}"
    )

In [ ]:
model.eval()

with torch.no_grad():

    predictions = model(X_test)

In [ ]:
predictions = target_scaler.inverse_transform(
    predictions.numpy()
)

actual = target_scaler.inverse_transform(
    y_test.numpy()
)

In [ ]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(
    mean_squared_error(
        actual,
        predictions
    )
)

print("RMSE:", rmse)